# Phase 1 Data Pipeline (MIND-small)

## Scope Lock
- Phase 1 only: Data Pipeline + Feature Engineering.
- MIND-small only. No MIND-large processing in this notebook.
- No causal estimation/refutation cells and no RL cells.

## Contract Locks From Documentation
- `A=1` for shown items; sampled controls use `A=0` and `Y_click=0`.
- `Y_diversity = 1 - cosine(U_history_emb, I_title_emb)`.
- Negative sampling target ratio: `1:4` (shown:negative).
- PCA requirement: `n_components=32` for tabular compatibility.

## User-Approved Overrides
- `U_history_emb` is locked to 768 dimensions for this implementation.
- Split strategy is by `ImpressionID` session boundary with `70/15/15` train/val/test.

## Outputs
- `data/scm_train.parquet`
- `data/scm_val.parquet`
- `data/scm_test.parquet`
- `data/processed/phase1_data_report.json`
- `data/interim/scm_full_embeddings.parquet` (full embeddings sidecar)


In [1]:
import sys
import os
import json
import logging
from pathlib import Path
import random
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Adjust context to project root
cwd = Path.cwd()
if cwd.name == "notebooks":
    root = cwd.parent
else:
    root = cwd

PROJECT_ROOT = root

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RAW_SMALL_DIR = RAW_DIR / "MIND-small"

RAW_SMALL_TRAIN_DIR = RAW_SMALL_DIR / "train"
RAW_SMALL_DEV_DIR = RAW_SMALL_DIR / "dev"
RAW_SMALL_TEST_DIR = RAW_SMALL_DIR / "test"

DATASET = "small"
SEED = 42
NEG_RATIO = 4
PCA_COMPONENTS = 32
TITLE_EMBED_DIM = 768
TITLE_EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
ENTITY_EMBED_DIM = 100
SPLIT_RATIOS = (0.70, 0.15, 0.15)
MAX_BEHAVIOR_ROWS = None  # None for full dataset, int for fast smoke tests.

for path in [
    DATA_DIR,
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    NOTEBOOKS_DIR,
    RAW_SMALL_DIR,
    RAW_SMALL_TRAIN_DIR,
    RAW_SMALL_DEV_DIR,
    RAW_SMALL_TEST_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output path: {NOTEBOOKS_DIR / 'phase_1_data_pipeline_mind_small.ipynb'}")
print(f"Data root: {DATA_DIR}")

Project root: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs
Notebook output path: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\notebooks\phase_1_data_pipeline_mind_small.ipynb
Data root: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data


## Function Library (Definitions First)

All helper functions are defined before execution to keep notebook flow deterministic and reproducible.

In [2]:
import ast
import hashlib
import importlib
import json
import os
import shutil
import urllib.request
import zipfile
from datetime import datetime, timezone
from typing import Callable, Dict, List, Sequence, Tuple

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import HashingVectorizer


def l2_normalize(vec: np.ndarray) -> np.ndarray:
    arr = np.asarray(vec, dtype=np.float32)
    norm = np.linalg.norm(arr)
    if norm == 0 or not np.isfinite(norm):
        return np.zeros_like(arr, dtype=np.float32)
    return (arr / norm).astype(np.float32)


def cosine_diversity(user_emb: np.ndarray, item_emb: np.ndarray) -> float:
    u = l2_normalize(user_emb)
    i = l2_normalize(item_emb)
    if u.size == 0 or i.size == 0:
        return 0.0
    cosine_sim = float(np.dot(u, i))
    if not np.isfinite(cosine_sim):
        cosine_sim = 0.0
    return float(np.clip(1.0 - cosine_sim, 0.0, 1.0))


def parse_history(history_str: str) -> List[str]:
    if pd.isna(history_str):
        return []
    text = str(history_str).strip()
    if not text:
        return []
    return [tok for tok in text.split() if tok]


def parse_impressions(impressions_str: str) -> List[Tuple[str, int]]:
    if pd.isna(impressions_str):
        return []
    pairs: List[Tuple[str, int]] = []
    for token in str(impressions_str).split():
        if "-" not in token:
            continue
        item_id, click_flag = token.rsplit("-", 1)
        if not item_id:
            continue
        pairs.append((item_id, 1 if click_flag == "1" else 0))
    return pairs


def parse_entities(entity_blob: str) -> List[str]:
    if pd.isna(entity_blob):
        return []
    text = str(entity_blob).strip()
    if not text or text in {"[]", "nan", "None"}:
        return []

    payload = None
    for parser in (json.loads, ast.literal_eval):
        try:
            payload = parser(text)
            break
        except Exception:
            continue

    if not isinstance(payload, list):
        return []

    ids: List[str] = []
    for node in payload:
        if isinstance(node, dict):
            entity_id = node.get("WikidataId") or node.get("WikidataID") or node.get("Label")
            if entity_id:
                ids.append(str(entity_id))
    return ids


def stable_hash_vector(text: str, dim: int) -> np.ndarray:
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    base = np.frombuffer(digest, dtype=np.uint8).astype(np.float32)
    repeats = int(np.ceil(dim / base.size))
    vec = np.tile(base, repeats)[:dim]
    vec = (vec - 127.5) / 127.5
    return l2_normalize(vec)


def mean_embeddings(vectors: List[np.ndarray], dim: int) -> np.ndarray:
    if not vectors:
        return np.zeros(dim, dtype=np.float32)
    matrix = np.vstack([np.asarray(v, dtype=np.float32) for v in vectors])
    return l2_normalize(matrix.mean(axis=0))


def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, destination)


def extract_zip(zip_path: Path, target_dir: Path) -> None:
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir)


def canonicalize_split_files(split_dir: Path) -> Dict[str, Path]:
    split_dir.mkdir(parents=True, exist_ok=True)
    news_candidates = sorted(split_dir.rglob("news.tsv"))
    behavior_candidates = sorted(split_dir.rglob("behaviors.tsv"))

    if not news_candidates or not behavior_candidates:
        return {}

    news_src = news_candidates[0]
    behavior_src = behavior_candidates[0]
    news_dst = split_dir / "news.tsv"
    behavior_dst = split_dir / "behaviors.tsv"

    if news_src.resolve() != news_dst.resolve():
        shutil.copy2(news_src, news_dst)
    if behavior_src.resolve() != behavior_dst.resolve():
        shutil.copy2(behavior_src, behavior_dst)

    return {"news": news_dst, "behaviors": behavior_dst}


def discover_split_pairs(root_dir: Path) -> Dict[str, Dict[str, Path]]:
    discovered: Dict[str, Dict[str, Path]] = {}
    if not root_dir.exists():
        return discovered

    for news_path in sorted(root_dir.rglob("news.tsv")):
        marker = news_path.parent.as_posix().lower()
        if "train" in marker:
            split = "train"
        elif "dev" in marker or "valid" in marker or "val" in marker:
            split = "dev"
        elif "test" in marker:
            split = "test"
        else:
            continue

        if split in discovered:
            continue

        behavior_path = news_path.parent / "behaviors.tsv"
        if not behavior_path.exists():
            alt_candidates = sorted(news_path.parent.rglob("behaviors.tsv"))
            if not alt_candidates:
                continue
            behavior_path = alt_candidates[0]

        discovered[split] = {"news": news_path, "behaviors": behavior_path}

    return discovered


def prepare_mind_small_dataset(raw_small_dir: Path, external_candidates: Sequence[Path]) -> Dict[str, Path]:
    split_dirs = {
        "train": raw_small_dir / "train",
        "dev": raw_small_dir / "dev",
        "test": raw_small_dir / "test",
    }

    resolved: Dict[str, Dict[str, Path]] = {}
    for split, split_dir in split_dirs.items():
        canonical = canonicalize_split_files(split_dir)
        if canonical:
            resolved[split] = canonical

    # Try extracting local zip files first.
    for candidate_root in external_candidates:
        if not candidate_root.exists():
            continue
        for zip_path in sorted(candidate_root.rglob("*.zip")):
            name = zip_path.name.lower()
            if "mindsmall" not in name and "mind_small" not in name:
                continue
            if "train" in name:
                split = "train"
            elif "dev" in name or "valid" in name or "val" in name:
                split = "dev"
            elif "test" in name:
                split = "test"
            else:
                continue
            extract_zip(zip_path, split_dirs[split])

    # Try env-based downloads if any split is still missing.
    env_urls = {
        "train": os.getenv("MIND_SMALL_TRAIN_URL", "").strip(),
        "dev": os.getenv("MIND_SMALL_DEV_URL", "").strip(),
        "test": os.getenv("MIND_SMALL_TEST_URL", "").strip(),
    }
    for split, url in env_urls.items():
        if not url:
            continue
        zip_destination = raw_small_dir / f"MINDsmall_{split}.zip"
        if not zip_destination.exists():
            download_file(url, zip_destination)
        extract_zip(zip_destination, split_dirs[split])

    # Copy from any pre-extracted candidate folders.
    for candidate_root in external_candidates:
        discovered = discover_split_pairs(candidate_root)
        for split, file_pair in discovered.items():
            split_dirs[split].mkdir(parents=True, exist_ok=True)
            news_dst = split_dirs[split] / "news.tsv"
            behavior_dst = split_dirs[split] / "behaviors.tsv"
            if file_pair["news"].resolve() != news_dst.resolve():
                shutil.copy2(file_pair["news"], news_dst)
            if file_pair["behaviors"].resolve() != behavior_dst.resolve():
                shutil.copy2(file_pair["behaviors"], behavior_dst)

    for split, split_dir in split_dirs.items():
        canonical = canonicalize_split_files(split_dir)
        if canonical:
            resolved[split] = canonical

    missing_required = [split for split in ("train", "dev") if split not in resolved]
    if missing_required:
        raise FileNotFoundError(
            "Missing required MIND-small splits. Expected train/dev news.tsv and behaviors.tsv. "
            "Place local files under data/raw/MIND-small/{train,dev}/ or provide "
            "MIND_SMALL_TRAIN_URL and MIND_SMALL_DEV_URL environment variables."
        )

    result = {
        "train_news": resolved["train"]["news"],
        "train_behaviors": resolved["train"]["behaviors"],
        "dev_news": resolved["dev"]["news"],
        "dev_behaviors": resolved["dev"]["behaviors"],
    }
    if "test" in resolved:
        result["test_news"] = resolved["test"]["news"]
        result["test_behaviors"] = resolved["test"]["behaviors"]
    return result


def load_news_frames(news_paths: Sequence[Path]) -> pd.DataFrame:
    columns = [
        "NewsID",
        "Category",
        "SubCategory",
        "Title",
        "Abstract",
        "URL",
        "TitleEntities",
        "AbstractEntities",
    ]
    frames = []
    for path in news_paths:
        frame = pd.read_csv(
            path,
            sep="\t",
            names=columns,
            header=None,
            dtype=str,
            keep_default_na=False,
            na_filter=False,
            encoding="utf-8",
        )
        frames.append(frame)

    news_df = pd.concat(frames, ignore_index=True)
    news_df["NewsID"] = news_df["NewsID"].astype(str)
    news_df = news_df.drop_duplicates(subset=["NewsID"], keep="first")
    return news_df.reset_index(drop=True)


def load_behavior_frames(behavior_paths: Sequence[Path]) -> pd.DataFrame:
    columns = ["ImpressionID", "UserID", "Time", "History", "Impressions"]
    frames = []
    for path in behavior_paths:
        frame = pd.read_csv(
            path,
            sep="\t",
            names=columns,
            header=None,
            dtype=str,
            keep_default_na=False,
            na_filter=False,
            encoding="utf-8",
        )
        frames.append(frame)

    behaviors_df = pd.concat(frames, ignore_index=True)
    behaviors_df["ImpressionID"] = behaviors_df["ImpressionID"].astype(str)
    behaviors_df["UserID"] = behaviors_df["UserID"].astype(str)
    return behaviors_df.reset_index(drop=True)


def build_title_encoder(model_name: str, expected_dim: int) -> Tuple[Callable[[List[str]], np.ndarray], Dict[str, str]]:
    try:
        st_module = importlib.import_module("sentence_transformers")
        sentence_transformer_cls = getattr(st_module, "SentenceTransformer")

        model = sentence_transformer_cls(model_name)

        def encode(texts: List[str], batch_size: int = 256) -> np.ndarray:
            embeddings = model.encode(
                texts,
                batch_size=batch_size,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
            )
            if embeddings.shape[1] != expected_dim:
                raise ValueError(
                    f"Expected {expected_dim}-dim title embeddings, got {embeddings.shape[1]} from {model_name}."
                )
            return embeddings.astype(np.float32)

        return encode, {
            "encoder_type": "sentence_transformers",
            "model": model_name,
            "embedding_dim": str(expected_dim),
        }
    except Exception as exc:
        vectorizer = HashingVectorizer(n_features=expected_dim, alternate_sign=False, norm=None)

        def encode(texts: List[str]) -> np.ndarray:
            sparse_matrix = vectorizer.transform([text if isinstance(text, str) else "" for text in texts]).astype(np.float32)
            dense = sparse_matrix.toarray()
            norms = np.linalg.norm(dense, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            return (dense / norms).astype(np.float32)

        return encode, {
            "encoder_type": "hashing_vectorizer_fallback",
            "model": "hashing",
            "embedding_dim": str(expected_dim),
            "fallback_reason": str(exc),
        }


def build_sentiment_analyzer():
    try:
        import nltk
        from nltk.sentiment import SentimentIntensityAnalyzer

        try:
            nltk.data.find("sentiment/vader_lexicon.zip")
        except LookupError:
            nltk.download("vader_lexicon", quiet=True)

        return SentimentIntensityAnalyzer()
    except Exception as exc:
        print(f"Sentiment analyzer unavailable, defaulting to 0.0 sentiment. Reason: {exc}")
        return None


def score_sentiment(text: str, analyzer) -> float:
    if analyzer is None:
        return 0.0
    try:
        return float(analyzer.polarity_scores(text if isinstance(text, str) else "")["compound"])
    except Exception:
        return 0.0


def compute_news_features(
    news_df: pd.DataFrame,
    title_encoder: Callable[[List[str]], np.ndarray],
    sentiment_analyzer,
    entity_dim: int,
) -> pd.DataFrame:
    titles = news_df["Title"].astype(str).tolist()
    title_embeddings = title_encoder(titles)
    if title_embeddings.shape[0] != news_df.shape[0]:
        raise ValueError("Title embedding count mismatch with news rows.")

    entity_embeddings: List[List[float]] = []
    for entity_blob in news_df["TitleEntities"].tolist():
        entity_ids = parse_entities(entity_blob)
        if entity_ids:
            vectors = [stable_hash_vector(entity_id, entity_dim) for entity_id in entity_ids]
            entity_vector = mean_embeddings(vectors, entity_dim)
        else:
            entity_vector = np.zeros(entity_dim, dtype=np.float32)
        entity_embeddings.append(entity_vector.tolist())

    sentiments = [score_sentiment(title, sentiment_analyzer) for title in titles]

    features_df = pd.DataFrame(
        {
            "item_id": news_df["NewsID"].astype(str),
            "I_category": news_df["Category"].replace("", "unknown").astype(str),
            "I_subcategory": news_df["SubCategory"].replace("", "unknown").astype(str),
            "I_sentiment": sentiments,
        }
    )
    features_df["I_title_emb_full"] = [row.astype(np.float32).tolist() for row in title_embeddings]
    features_df["I_entity_emb_full"] = entity_embeddings
    features_df = features_df.drop_duplicates(subset=["item_id"], keep="first")
    return features_df.set_index("item_id", drop=True)


def prepare_behaviors(behaviors_df: pd.DataFrame, max_rows: int = None) -> pd.DataFrame:
    prepared = behaviors_df.copy()
    if max_rows is not None:
        prepared = prepared.head(int(max_rows)).copy()

    prepared["history_ids"] = prepared["History"].apply(parse_history)
    prepared["parsed_impressions"] = prepared["Impressions"].apply(parse_impressions)
    prepared["impression_count"] = prepared["parsed_impressions"].apply(len)
    prepared = prepared[prepared["impression_count"] > 0].copy()
    return prepared.reset_index(drop=True)


def build_session_user_features(
    behaviors_df: pd.DataFrame,
    news_features_df: pd.DataFrame,
    title_dim: int,
) -> pd.DataFrame:
    title_lookup = news_features_df["I_title_emb_full"].to_dict()
    records = []

    for row in behaviors_df.itertuples(index=False):
        history_vectors = [
            np.asarray(title_lookup[item_id], dtype=np.float32)
            for item_id in row.history_ids
            if item_id in title_lookup
        ]
        u_history_emb = mean_embeddings(history_vectors, title_dim)
        records.append(
            {
                "impression_id": str(row.ImpressionID),
                "user_id": str(row.UserID),
                "U_history_emb_full": u_history_emb.tolist(),
                "U_dwell_mean": float(row.impression_count),
                "U_click_count": float(len(row.history_ids)),
            }
        )

    return pd.DataFrame(records)


def sample_negative_items(item_pool: np.ndarray, shown_item_ids: Sequence[str], count: int, rng) -> List[str]:
    if count <= 0:
        return []
    shown_set = set(shown_item_ids)
    candidate_pool = [item_id for item_id in item_pool if item_id not in shown_set]
    if not candidate_pool:
        return []

    replace = count > len(candidate_pool)
    sampled = rng.choice(candidate_pool, size=count, replace=replace)
    return [str(item_id) for item_id in sampled.tolist()]


def build_scm_dataframe(
    behaviors_df: pd.DataFrame,
    news_features_df: pd.DataFrame,
    session_user_features_df: pd.DataFrame,
    neg_ratio: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    news_lookup = news_features_df.to_dict("index")
    session_lookup = session_user_features_df.set_index("impression_id").to_dict("index")
    item_pool = np.array(news_features_df.index.tolist())

    records = []
    for row in behaviors_df.itertuples(index=False):
        impression_id = str(row.ImpressionID)
        if impression_id not in session_lookup:
            continue

        session_features = session_lookup[impression_id]
        u_history_emb = np.asarray(session_features["U_history_emb_full"], dtype=np.float32)

        shown_items: List[str] = []
        for item_id, clicked in row.parsed_impressions:
            item_features = news_lookup.get(item_id)
            if item_features is None:
                continue

            shown_items.append(item_id)
            i_title_emb = np.asarray(item_features["I_title_emb_full"], dtype=np.float32)

            records.append(
                {
                    "user_id": str(row.UserID),
                    "impression_id": impression_id,
                    "time": str(row.Time),
                    "item_id": str(item_id),
                    "A": 1,
                    "Y_click": int(clicked),
                    "Y_diversity": cosine_diversity(u_history_emb, i_title_emb),
                    "U_dwell_mean": float(session_features["U_dwell_mean"]),
                    "U_click_count": float(session_features["U_click_count"]),
                    "I_category": str(item_features["I_category"]),
                    "I_subcategory": str(item_features["I_subcategory"]),
                    "I_sentiment": float(item_features["I_sentiment"]),
                    "U_history_emb_full": u_history_emb.tolist(),
                    "I_title_emb_full": i_title_emb.tolist(),
                    "I_entity_emb_full": list(item_features["I_entity_emb_full"]),
                }
            )

        negative_count = len(shown_items) * int(neg_ratio)
        sampled_negatives = sample_negative_items(item_pool, shown_items, negative_count, rng)
        for item_id in sampled_negatives:
            item_features = news_lookup.get(item_id)
            if item_features is None:
                continue

            i_title_emb = np.asarray(item_features["I_title_emb_full"], dtype=np.float32)
            records.append(
                {
                    "user_id": str(row.UserID),
                    "impression_id": impression_id,
                    "time": str(row.Time),
                    "item_id": str(item_id),
                    "A": 0,
                    "Y_click": 0,
                    "Y_diversity": cosine_diversity(u_history_emb, i_title_emb),
                    "U_dwell_mean": float(session_features["U_dwell_mean"]),
                    "U_click_count": float(session_features["U_click_count"]),
                    "I_category": str(item_features["I_category"]),
                    "I_subcategory": str(item_features["I_subcategory"]),
                    "I_sentiment": float(item_features["I_sentiment"]),
                    "U_history_emb_full": u_history_emb.tolist(),
                    "I_title_emb_full": i_title_emb.tolist(),
                    "I_entity_emb_full": list(item_features["I_entity_emb_full"]),
                }
            )

    if not records:
        raise ValueError("SCM dataframe is empty. Check MIND-small extraction and parsing logic.")

    scm_df = pd.DataFrame.from_records(records)
    scm_df.insert(0, "row_id", np.arange(scm_df.shape[0], dtype=np.int64))
    scm_df["A"] = scm_df["A"].astype(np.int8)
    scm_df["Y_click"] = scm_df["Y_click"].astype(np.int8)
    scm_df["Y_diversity"] = scm_df["Y_diversity"].astype(np.float32)
    scm_df["U_dwell_mean"] = scm_df["U_dwell_mean"].astype(np.float32)
    scm_df["U_click_count"] = scm_df["U_click_count"].astype(np.float32)
    scm_df["I_sentiment"] = scm_df["I_sentiment"].astype(np.float32)
    return scm_df


def reduce_embedding_columns(scm_df: pd.DataFrame, n_components: int, seed: int) -> Tuple[pd.DataFrame, Dict[str, Dict[str, float]]]:
    reduced_df = scm_df.copy()
    pca_report: Dict[str, Dict[str, float]] = {}

    specs = [
        ("U_history_emb_full", "U_pca"),
        ("I_entity_emb_full", "I_entity_pca"),
    ]

    for source_col, prefix in specs:
        matrix = np.vstack(reduced_df[source_col].apply(lambda row: np.asarray(row, dtype=np.float32)).to_numpy())
        components = int(min(n_components, matrix.shape[0], matrix.shape[1]))
        if components <= 0:
            raise ValueError(f"Cannot run PCA on column {source_col}; matrix shape={matrix.shape}.")

        pca = PCA(n_components=components, random_state=seed)
        transformed = pca.fit_transform(matrix).astype(np.float32)

        for idx in range(components):
            reduced_df[f"{prefix}_{idx}"] = transformed[:, idx]
        for idx in range(components, n_components):
            reduced_df[f"{prefix}_{idx}"] = 0.0

        pca_report[prefix] = {
            "n_components": components,
            "explained_variance_ratio_sum": float(np.sum(pca.explained_variance_ratio_)),
        }

    return reduced_df, pca_report


def split_by_impression_id(
    scm_df: pd.DataFrame,
    split_ratios: Tuple[float, float, float],
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, int]]:
    train_ratio, val_ratio, test_ratio = split_ratios
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("Split ratios must sum to 1.0")

    keys = scm_df["impression_id"].astype(str).unique().tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(keys)

    n_total = len(keys)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    train_keys = set(keys[:n_train])
    val_keys = set(keys[n_train:n_train + n_val])
    test_keys = set(keys[n_train + n_val:])

    train_df = scm_df[scm_df["impression_id"].isin(train_keys)].copy()
    val_df = scm_df[scm_df["impression_id"].isin(val_keys)].copy()
    test_df = scm_df[scm_df["impression_id"].isin(test_keys)].copy()

    if train_keys & val_keys or train_keys & test_keys or val_keys & test_keys:
        raise AssertionError("Leakage detected: overlapping ImpressionID keys across splits.")

    split_report = {
        "total_impressions": n_total,
        "train_impressions": len(train_keys),
        "val_impressions": len(val_keys),
        "test_impressions": len(test_keys),
    }
    return train_df, val_df, test_df, split_report


def save_parquet(df: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(output_path, index=False)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to save parquet at {output_path}. Install pyarrow or fastparquet."
        ) from exc


def run_quality_checks(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    required_cols: Sequence[str],
    neg_ratio: int,
) -> Dict[str, object]:
    combined = pd.concat([train_df, val_df, test_df], ignore_index=True)

    missing_columns = [col for col in required_cols if col not in combined.columns]
    critical_cols = [
        "user_id",
        "item_id",
        "impression_id",
        "A",
        "Y_click",
        "Y_diversity",
        "U_dwell_mean",
        "I_category",
        "I_sentiment",
    ]

    null_summary = {col: int(combined[col].isna().sum()) for col in critical_cols if col in combined.columns}
    y_click_values = sorted(set(combined["Y_click"].dropna().astype(int).tolist())) if "Y_click" in combined.columns else []
    y_click_binary = set(y_click_values).issubset({0, 1})

    y_div_min = float(combined["Y_diversity"].min()) if "Y_diversity" in combined.columns else None
    y_div_max = float(combined["Y_diversity"].max()) if "Y_diversity" in combined.columns else None
    y_div_complete = bool(combined["Y_diversity"].notna().all()) if "Y_diversity" in combined.columns else False

    treatment_count = int((combined["A"] == 1).sum()) if "A" in combined.columns else 0
    control_count = int((combined["A"] == 0).sum()) if "A" in combined.columns else 0
    observed_ratio = float(control_count / treatment_count) if treatment_count > 0 else None
    ratio_ok = (
        observed_ratio is not None and abs(observed_ratio - neg_ratio) <= (0.35 * neg_ratio)
    )

    train_keys = set(train_df["impression_id"].astype(str).unique())
    val_keys = set(val_df["impression_id"].astype(str).unique())
    test_keys = set(test_df["impression_id"].astype(str).unique())
    leakage_ok = not (train_keys & val_keys or train_keys & test_keys or val_keys & test_keys)

    report = {
        "missing_columns": missing_columns,
        "null_summary": null_summary,
        "y_click_values": y_click_values,
        "y_click_binary": y_click_binary,
        "y_diversity_min": y_div_min,
        "y_diversity_max": y_div_max,
        "y_diversity_complete": y_div_complete,
        "treatment_count": treatment_count,
        "control_count": control_count,
        "observed_control_per_treatment": observed_ratio,
        "target_control_per_treatment": float(neg_ratio),
        "ratio_check_pass": ratio_ok,
        "split_leakage_check_pass": leakage_ok,
    }

    if missing_columns:
        raise AssertionError(f"Missing required columns: {missing_columns}")
    if sum(null_summary.values()) > 0:
        raise AssertionError(f"Critical nulls found: {null_summary}")
    if not y_click_binary:
        raise AssertionError(f"Y_click is not binary. Values={y_click_values}")
    if not y_div_complete:
        raise AssertionError("Y_diversity contains null values.")
    if y_div_min is None or y_div_max is None or y_div_min < 0.0 or y_div_max > 1.0:
        raise AssertionError(f"Y_diversity out of expected range [0, 1]: min={y_div_min}, max={y_div_max}")
    if not leakage_ok:
        raise AssertionError("Split leakage detected by ImpressionID.")

    return report


def build_phase1_report(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    split_report: Dict[str, int],
    quality_report: Dict[str, object],
    pca_report: Dict[str, Dict[str, float]],
    encoder_meta: Dict[str, str],
    output_paths: Dict[str, str],
) -> Dict[str, object]:
    all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": {
            "dataset": DATASET,
            "seed": SEED,
            "neg_ratio": NEG_RATIO,
            "pca_components": PCA_COMPONENTS,
            "title_embedding_dim": TITLE_EMBED_DIM,
            "split_ratios": list(SPLIT_RATIOS),
            "split_key": "impression_id",
        },
        "encoder": encoder_meta,
        "rows": {
            "train": int(train_df.shape[0]),
            "val": int(val_df.shape[0]),
            "test": int(test_df.shape[0]),
            "total": int(all_df.shape[0]),
        },
        "class_balance": {
            "A_counts": {str(k): int(v) for k, v in all_df["A"].value_counts(dropna=False).to_dict().items()},
            "Y_click_counts": {str(k): int(v) for k, v in all_df["Y_click"].value_counts(dropna=False).to_dict().items()},
        },
        "null_summary": {col: int(all_df[col].isna().sum()) for col in ["A", "Y_click", "Y_diversity", "U_dwell_mean", "I_sentiment"]},
        "split_summary": split_report,
        "pca": pca_report,
        "quality_checks": quality_report,
        "outputs": output_paths,
    }

In [3]:
# Hotfix: MIND-small ImpressionID values can overlap across splits.
# We define a stable session key as "<split_source>:<ImpressionID>" for uniqueness.
import gc

def _infer_split_source(path: Path) -> str:
    marker = path.parent.as_posix().lower()
    if "train" in marker:
        return "train"
    if "dev" in marker or "valid" in marker or "val" in marker:
        return "dev"
    if "test" in marker:
        return "test"
    return "unknown"

def hash_split(session_key: str, ratios: Tuple[float, float, float]) -> str:
    """Deterministic hash-based split into train/val/test."""
    import hashlib
    val = int(hashlib.md5(session_key.encode('utf-8')).hexdigest(), 16) % 100
    train_thresh = int(ratios[0] * 100)
    val_thresh = train_thresh + int(ratios[1] * 100)
    if val < train_thresh: return "train"
    elif val < val_thresh: return "val"
    return "test"

def process_behavior_chunk(
    behaviors_df: pd.DataFrame,
    news_features_df: pd.DataFrame,
    neg_ratio: int,
    seed: int,
    split_source: str
) -> pd.DataFrame:
    # 1. Clean & Parse
    behaviors_df["SplitSource"] = split_source
    behaviors_df["SessionKey"] = behaviors_df["SplitSource"] + ":" + behaviors_df["ImpressionID"]
    behaviors_df = prepare_behaviors(behaviors_df, max_rows=MAX_BEHAVIOR_ROWS)
    if behaviors_df.empty:
        return pd.DataFrame()
        
    # 2. Build session features
    session_user_features_df = build_session_user_features(
        behaviors_df=behaviors_df,
        news_features_df=news_features_df,
        title_dim=TITLE_EMBED_DIM,
    )
    
    # 3. Build SCM rows
    scm_df = build_scm_dataframe(
        behaviors_df=behaviors_df,
        news_features_df=news_features_df,
        session_user_features_df=session_user_features_df,
        neg_ratio=neg_ratio,
        seed=seed,
    )
    return scm_df

def stream_and_build(
    behavior_paths: Sequence[Path],
    news_features_df: pd.DataFrame,
    output_dir: Path,
    chunksize: int = 2000,
    neg_ratio: int = 4,
    seed: int = 42
):
    output_dir.mkdir(parents=True, exist_ok=True)
    columns = ["ImpressionID", "UserID", "Time", "History", "Impressions"]
    
    part_idx = 0
    stats = {"train": 0, "val": 0, "test": 0, "total_rows": 0}
    
    for path in behavior_paths:
        split_source = _infer_split_source(Path(path))
        reader = pd.read_csv(
            path, sep="\t", names=columns, header=None,
            dtype=str, keep_default_na=False, na_filter=False,
            encoding="utf-8", chunksize=chunksize
        )
        
        for beh_chunk in reader:
            scm_chunk = process_behavior_chunk(beh_chunk, news_features_df, neg_ratio, seed + part_idx, split_source)
            if scm_chunk.empty:
                continue
                
            # Assign splits by hash
            scm_chunk["split_bucket"] = scm_chunk["impression_id"].apply(lambda x: hash_split(x, SPLIT_RATIOS))
            
            for bucket in ["train", "val", "test"]:
                bucket_df = scm_chunk[scm_chunk["split_bucket"] == bucket].copy()
                if not bucket_df.empty:
                    bucket_df = bucket_df.drop(columns=["split_bucket"])
                    out_path = output_dir / f"scm_{bucket}" / f"part_{part_idx:05d}.parquet"
                    save_parquet(bucket_df, out_path)
                    stats[bucket] += len(bucket_df)
                    stats["total_rows"] += len(bucket_df)
                    
            part_idx += 1
            del beh_chunk, scm_chunk
            gc.collect()
            
    return stats

## Execution (Top-Down Deterministic Flow)

Run all execution cells from top to bottom with fixed seed.

### 1_load_news

In [4]:
external_candidates = [
    PROJECT_ROOT / "MIND small dataset",
]

dataset_files = prepare_mind_small_dataset(RAW_SMALL_DIR, external_candidates)
print("Resolved MIND-small files:")
for key, value in dataset_files.items():
    print(f"  {key}: {value}")

news_paths = [dataset_files["train_news"], dataset_files["dev_news"]]
if "test_news" in dataset_files:
    news_paths.append(dataset_files["test_news"])

news_df = load_news_frames(news_paths)
print(f"News rows: {news_df.shape[0]:,} | Unique items: {news_df['NewsID'].nunique():,}")
news_df.head(3)

Resolved MIND-small files:
  train_news: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\raw\MIND-small\train\news.tsv
  train_behaviors: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\raw\MIND-small\train\behaviors.tsv
  dev_news: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\raw\MIND-small\dev\news.tsv
  dev_behaviors: c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\raw\MIND-small\dev\behaviors.tsv
News rows: 65,238 | Unique items: 65,238


,NewsID,Category,SubCategory,Title,Abstract,URL,TitleEntities,AbstractEntities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."


### 2_news_features

In [5]:
title_encoder, encoder_meta = build_title_encoder(TITLE_EMBED_MODEL, TITLE_EMBED_DIM)
sentiment_analyzer = build_sentiment_analyzer()

news_features_df = compute_news_features(
    news_df=news_df,
    title_encoder=title_encoder,
    sentiment_analyzer=sentiment_analyzer,
    entity_dim=ENTITY_EMBED_DIM,
)

print("Encoder metadata:")
print(json.dumps(encoder_meta, indent=2))
print(f"News feature rows: {news_features_df.shape[0]:,}")
news_features_df.head(3)

c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: No device provided, using cpu
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transf

Encoder metadata:
{
  "encoder_type": "sentence_transformers",
  "model": "sentence-transformers/all-mpnet-base-v2",
  "embedding_dim": "768"
}
News feature rows: 65,238


,I_category,I_subcategory,I_sentiment,I_title_emb_full,I_entity_emb_full
item_id,,,,,
N55528,lifestyle,lifestyleroyals,-0.0516,"[0.01909557729959488, -0.004881767090409994, -...","[0.0644809827208519, 0.031125439330935478, -0...."
N19639,health,weightloss,-0.6249,"[0.06197544187307358, 0.04211072996258736, 0.0...","[0.11611241847276688, 0.09686726331710815, 0.1..."
N61837,news,newsworld,-0.5719,"[-0.01050851121544838, 0.0733732208609581, 0.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


### 3_load_behaviors

In [ ]:
behavior_paths = [dataset_files["train_behaviors"], dataset_files["dev_behaviors"]]
if "test_behaviors" in dataset_files:
    behavior_paths.append(dataset_files["test_behaviors"])

scm_parts_dir = DATA_DIR / "scm_parts"
if scm_parts_dir.exists():
    import shutil
    shutil.rmtree(scm_parts_dir)
    
print(f"Streaming behaviors to partitioned parquet at {scm_parts_dir}...")
stats = stream_and_build(
    behavior_paths=behavior_paths,
    news_features_df=news_features_df,
    output_dir=scm_parts_dir,
    chunksize=2000,
    neg_ratio=NEG_RATIO,
    seed=SEED
)

print("Streaming build complete!")
print(json.dumps(stats, indent=2))

Streaming behaviors to partitioned parquet at c:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\scm_parts...


### 4_inspect_partitioned_dataset

In [12]:
import pyarrow.dataset as ds

dataset = ds.dataset(scm_parts_dir)
print(f"Total rows in partitioned dataset: {dataset.count_rows():,}")
print(f"Total parquet parts: {len(dataset.files)}")

# Preview chunk
dataset.to_table(limit=3).to_pandas()

Session-level user feature rows: 230,117


,impression_id,raw_impression_id,split_source,user_id,U_history_emb_full,U_dwell_mean,U_click_count
0,train:1,train:1,train,U13740,"[-0.007266818545758724, 0.08281856030225754, 0...",2.0,9.0
1,train:2,train:2,train,U91836,"[0.019121352583169937, 0.11352706700563431, 0....",11.0,82.0
2,train:3,train:3,train,U73700,"[-0.03624367341399193, 0.06310753524303436, -0...",36.0,16.0


## Phase 1 Outputs

In [ ]:
print("Primary notebook:")
print(f"  {NOTEBOOKS_DIR / 'phase_1_data_pipeline_mind_small.ipynb'}")

print("\nArtifacts (Partitioned Parquet):")
print(f"  {scm_parts_dir}")

print("\nDataset stats:")
print(json.dumps(stats, indent=2))

MemoryError: Unable to allocate 1.49 MiB for an array with shape (65236,) and data type <U6

: 

In [33]:
print(json.dumps(quality_report, indent=2))

print("\nSplit sizes (rows):")
print(f"  train: {train_df.shape[0]:,}")
print(f"  val:   {val_df.shape[0]:,}")
print(f"  test:  {test_df.shape[0]:,}")

print("\nTreatment ratio check (A=0 per A=1):")
observed_ratio = quality_report.get("observed_control_per_treatment")
print(f"  observed: {observed_ratio}")
print(f"  target:   {NEG_RATIO}")

print("\nSample preview: train")
train_df.head(5)

{
  "missing_columns": [],
  "null_summary": {
    "user_id": 0,
    "item_id": 0,
    "impression_id": 0,
    "A": 0,
    "Y_click": 0,
    "Y_diversity": 0,
    "U_dwell_mean": 0,
    "I_category": 0,
    "I_sentiment": 0
  },
  "y_click_values": [
    0,
    1
  ],
  "y_click_binary": true,
  "y_diversity_min": 0.1643136739730835,
  "y_diversity_max": 1.0,
  "y_diversity_complete": true,
  "treatment_count": 17831,
  "control_count": 71324,
  "observed_control_per_treatment": 4.0,
  "target_control_per_treatment": 4.0,
  "ratio_check_pass": true,
  "split_leakage_check_pass": true
}

Split sizes (rows):
  train: 60,410
  val:   14,185
  test:  14,560

Treatment ratio check (A=0 per A=1):
  observed: 4.0
  target:   4

Sample preview: train


,row_id,user_id,impression_id,raw_impression_id,split_source,time,item_id,A,Y_click,Y_diversity,...,I_entity_pca_22,I_entity_pca_23,I_entity_pca_24,I_entity_pca_25,I_entity_pca_26,I_entity_pca_27,I_entity_pca_28,I_entity_pca_29,I_entity_pca_30,I_entity_pca_31
0,0,U13740,train:1,train:1,train,11/11/2019 9:05:58 AM,N55689,1,1,0.751154,...,0.139755,0.135783,0.203184,0.199903,0.103343,-0.094672,-0.153921,-0.014817,-0.020728,0.080984
1,1,U13740,train:1,train:1,train,11/11/2019 9:05:58 AM,N35729,1,0,0.893852,...,0.028782,-0.016340,0.039309,-0.144501,-0.064868,-0.028538,0.040636,0.127037,0.126725,0.091152
2,2,U13740,train:1,train:1,train,11/11/2019 9:05:58 AM,N39693,0,0,0.805327,...,0.004047,-0.008053,0.002478,-0.001687,-0.001029,-0.008832,-0.000514,-0.005524,-0.002918,0.002779
3,3,U13740,train:1,train:1,train,11/11/2019 9:05:58 AM,N47057,0,0,0.854998,...,-0.029234,-0.048455,-0.406758,-0.030183,0.028883,-0.031199,-0.158322,-0.216549,0.010816,-0.156312
4,4,U13740,train:1,train:1,train,11/11/2019 9:05:58 AM,N12123,0,0,0.975558,...,0.004047,-0.008053,0.002478,-0.001687,-0.001029,-0.008832,-0.000514,-0.005524,-0.002918,0.002779


## Phase 1 Outputs

In [34]:
print("Primary notebook:")
print(f"  {NOTEBOOKS_DIR / 'phase_1_data_pipeline_mind_small.ipynb'}")

print("\nArtifacts:")
for key, value in phase1_report["outputs"].items():
    print(f"  {key}: {value}")

print("\nDataset stats:")
print(json.dumps(phase1_report["rows"], indent=2))
print(json.dumps(phase1_report["class_balance"], indent=2))

Primary notebook:
  C:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\notebooks\phase_1_data_pipeline_mind_small.ipynb

Artifacts:
  scm_train_parquet: C:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\scm_train.parquet
  scm_val_parquet: C:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\scm_val.parquet
  scm_test_parquet: C:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\scm_test.parquet
  phase1_report_json: C:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\processed\phase1_data_report.json
  full_embeddings_sidecar: C:\Users\abhis\Code\Programs\Personal-Projects\causal_rs\data\interim\scm_full_embeddings.parquet

Dataset stats:
{
  "train": 60410,
  "val": 14185,
  "test": 14560,
  "total": 89155
}
{
  "A_counts": {
    "0": 71324,
    "1": 17831
  },
  "Y_click_counts": {
    "0": 88438,
    "1": 717
  }
}
